In [1]:
import os, sys
project_root = os.path.dirname(os.getcwd())
# print(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from src.data_ingestion import load_processed_data_from_gcs

sns.set_theme(style="whitegrid")
PALETTE = {"positive": "#2ecc71", "neutral": "#ea890b", "negative": "#e90e07"}



In [2]:
# load the data which is already preprocessed.
df = load_processed_data_from_gcs("processed_reviews.csv")
df = df.dropna(subset=["cleaned_review"]).reset_index(drop=True)

print(df.shape)


(17321, 4)


In [4]:
df['sentiments'].value_counts()

sentiments
positive    9503
neutral     6284
negative    1534
Name: count, dtype: int64

In [5]:
le = LabelEncoder()
df["label"] = le.fit_transform(df['sentiments'])
print("Label encoding..")
print(dict(zip(le.classes_, le.transform(le.classes_))))

Label encoding..
{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [6]:
df

,sentiments,cleaned_review,review_score,word_count,label
0,positive,i wish would have gotten one earlier love it a...,5.0,19,2
1,neutral,i ve learned this lesson again open the packag...,1.0,88,1
2,neutral,it is so slow and lags find better option,2.0,9,1
3,neutral,roller ball stopped working within months of m...,1.0,12,1
4,neutral,i like the color and size but it few days out ...,1.0,21,1
...,...,...,...,...,...
17316,positive,i love this speaker and love can take it anywh...,5.0,30,2
17317,positive,i use it in my house easy to connect and loud ...,4.0,13,2
17318,positive,the bass is good and the battery is amazing mu...,5.0,41,2
17319,positive,love it,5.0,2,2


In [8]:
X_text = df['cleaned_review']
X_num = df[["review_score", "word_count"]]
y = df["label"]

In [9]:
X_tr_text, X_te_text, X_tr_num, X_te_num, y_train, y_test = train_test_split(X_text, X_num, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
X_tr_text.shape, X_tr_num.shape, y_train.shape, y_test.shape, X_te_text.shape, X_te_num.shape

((13856,), (13856, 2), (13856,), (3465,), (3465,), (3465, 2))

In [19]:
# Vectorization  ----> Count Vectorizer -----> also called BAG OF WORDS
# documents = ["it felt like it would break easily and the texture of the mouse would get dirty quickly", "I am not very happy with the product"]
# {"it": 2, "felt": 1, "would": 2, "easily": 1, "and": 1, "the": 2, "texture": 1, "of": 1, "mouse": 1, "get": 1, "dirty": 1, "quickly": 1}
# cv = CountVectorizer()
# feature_matrix = cv.fit_transform(documents)
# print(feature_matrix.toarray())

cv = CountVectorizer(max_features=10000, ngram_range=(1, 2))
X_tr_cv = cv.fit_transform(X_tr_text)
X_te_cv = cv.transform(X_te_text)











In [23]:
len(cv.vocabulary_.keys())

10000

In [24]:
X_te_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 134099 stored elements and shape (3465, 10000)>

In [25]:
X_te_cv.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(3465, 10000))